In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
corruption_bools = [True, False, False, True, True]

SQL query used:
```SQL
SELECT TOP 500000
    sp.ra,
    sp.dec,

    sp.modelMag_u,
    sp.modelMag_g,
    sp.modelMag_r,
    sp.modelMag_i,
    sp.modelMag_z,

    sp.extinction_u,
    sp.extinction_g,
    sp.extinction_r,
    sp.extinction_i,
    sp.extinction_z,

    sp.modelMagErr_u,
    sp.modelMagErr_g,
    sp.modelMagErr_r,
    sp.modelMagErr_i,
    sp.modelMagErr_z,

    p.petroR50_r,
    p.petroR90_r,

    sp.z AS z_spec,
    sp.zErr AS z_spec_err,

    sp.plate,
    sp.mjd,
    sp.fiberID

FROM SpecPhoto AS sp
INNER JOIN PhotoObj AS p
    ON p.objID = sp.objID

WHERE
    sp.class = 'GALAXY'
    AND sp.zWarning = 0
    AND sp.sciencePrimary = 1
    AND p.clean = 1

    AND sp.z BETWEEN 0.01 AND 0.80
    AND sp.modelMag_r BETWEEN 14.0 AND 22.5

    AND sp.modelMagErr_u > 0.0
    AND sp.modelMagErr_u < 1.0
    AND sp.modelMagErr_g > 0.0
    AND sp.modelMagErr_g < 1.0
    AND sp.modelMagErr_r > 0.0
    AND sp.modelMagErr_r < 1.0
    AND sp.modelMagErr_i > 0.0
    AND sp.modelMagErr_i < 1.0
    AND sp.modelMagErr_z > 0.0
    AND sp.modelMagErr_z < 1.0
```

In [3]:
# Load the raw data from the CSV file downloaded from the SDSS website
raw_data = pd.read_csv('data/raw_data.csv', comment="#")
# Fix the seed for reproducibility
np.random.seed(42)
# Only keep N rows
raw_data = raw_data.sample(n=100000, random_state=42).reset_index(drop=True)

In [4]:
raw_data.keys()

Index(['ra', 'dec', 'modelMag_u', 'modelMag_g', 'modelMag_r', 'modelMag_i',
       'modelMag_z', 'extinction_u', 'extinction_g', 'extinction_r',
       'extinction_i', 'extinction_z', 'modelMagErr_u', 'modelMagErr_g',
       'modelMagErr_r', 'modelMagErr_i', 'modelMagErr_z', 'petroR50_r',
       'petroR90_r', 'z_spec', 'z_spec_err', 'plate', 'mjd', 'fiberID'],
      dtype='str')

In [5]:
raw_data = raw_data.reset_index(drop=True)
raw_data.insert(0, "source_id", raw_data.index.astype("int64"))

In [6]:
raw_data

,source_id,ra,dec,modelMag_u,modelMag_g,modelMag_r,modelMag_i,modelMag_z,extinction_u,extinction_g,...,modelMagErr_r,modelMagErr_i,modelMagErr_z,petroR50_r,petroR90_r,z_spec,z_spec_err,plate,mjd,fiberID
0,0,3.172060,16.225965,20.08005,19.46605,18.89523,18.59497,18.47980,0.269457,0.209959,...,0.013428,0.016253,0.053698,1.182304,2.648756,0.242301,0.000007,752,52251,366
1,1,171.916880,58.673318,23.13237,22.21313,21.31596,21.18022,21.28397,0.066990,0.052198,...,0.074143,0.083581,0.298449,0.826920,1.518105,0.464847,0.000046,8176,57131,564
2,2,176.616310,61.836570,23.98152,21.74346,20.19427,19.32727,19.00012,0.108331,0.084411,...,0.031744,0.021195,0.060592,0.980991,2.506225,0.599866,0.000134,7106,56663,433
3,3,46.312797,0.625693,21.52985,20.84500,19.97012,19.52558,19.29880,0.329318,0.256602,...,0.026210,0.026962,0.070927,0.881451,2.428902,0.246866,0.000013,709,52205,592
4,4,132.056780,37.191262,23.16212,22.65009,21.03706,19.80841,19.37544,0.131074,0.102132,...,0.070264,0.034517,0.083207,1.420430,2.755985,0.637921,0.000209,4609,56251,729
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,99995,143.524980,51.182413,22.70759,20.66292,18.89187,18.05602,17.64744,0.055158,0.042979,...,0.017017,0.012798,0.031039,1.683927,4.025187,0.482027,0.000076,767,52252,32
99996,99996,132.701060,29.322085,22.69968,21.74975,19.97836,19.11364,18.66771,0.156422,0.121883,...,0.032520,0.022651,0.062003,1.467812,3.953760,0.474225,0.000106,5183,55976,11
99997,99997,257.157870,34.279171,22.25893,20.44811,18.51361,17.78013,17.33319,0.102248,0.079671,...,0.015241,0.012739,0.027834,1.975193,4.893267,0.411120,0.000127,974,52427,109
99998,99998,227.546480,-1.246315,19.10097,17.82475,17.27918,16.99789,16.81784,0.313293,0.244116,...,0.009005,0.010209,0.030350,4.768658,10.619100,0.036534,0.000011,311,51665,127


In [7]:
# Split into train and test sets
train_data = raw_data.sample(frac=0.8, random_state=42)
test_data = raw_data.drop(train_data.index)

In [8]:
# Add train vs valid bool to train_data
f_valid = 0.2
train_data["is_valid"] = np.random.rand(len(train_data)) < f_valid

In [9]:
# Apply corruption only to training (and not to validation) for simplicity.
# Corruption 1: Introduce NaN values in the 'z_spec' column for 1% of the rows + large unreasonable values for 1% of the rows
if corruption_bools[0]:
    num_rows = len(train_data[train_data["is_valid"] == False])
    num_nan = int(0.01 * num_rows)
    num_inf = int(0.01 * num_rows)
    nan_indices = np.random.choice(train_data.index[train_data["is_valid"] == False], size=num_nan, replace=False)
    inf_indices = np.random.choice(train_data.index[train_data["is_valid"] == False], size=num_inf, replace=False)
    train_data.loc[nan_indices, 'z_spec'] = np.nan
    train_data.loc[inf_indices, 'z_spec'] = np.random.uniform(low=10, high=100, size=len(inf_indices))

In [10]:
# Corruption 2: Introduce outliers in the 'z_spec' column by adding random large values to 0.5% of the rows
if corruption_bools[1]:
    num_outliers = int(0.005 * num_rows)
    outlier_indices = np.random.choice(train_data.index[train_data["is_valid"] == False], size=num_outliers, replace=False)
    # Add random large values (e.g., between 5 and 10) to the 'z_spec' column for the selected outlier rows
    train_data.loc[outlier_indices, 'z_spec'] += np.random.uniform(5, 10, size=num_outliers)

In [11]:
# Corruption 3: Shuffle the 'z_spec' column to break the relationship between features and target for 0.5% of the rows
if corruption_bools[2]:
    num_shuffle = int(0.005 * num_rows)
    shuffle_indices = np.random.choice(train_data.index[train_data["is_valid"] == False], size=num_shuffle, replace=False)
    # Shuffle the 'z_spec' values for the selected rows
    shuffled_z_spec_values = train_data.loc[shuffle_indices, 'z_spec'].sample(frac=1).values
    train_data.loc[shuffle_indices, 'z_spec'] = shuffled_z_spec_values

In [12]:
# Corruption 4: negative values in the 'z_spec' column for 0.5% of the rows
if corruption_bools[3]:
    num_negative_errors = int(0.005 * num_rows)
    negative_error_indices = np.random.choice(train_data.index[train_data["is_valid"] == False], size=num_negative_errors, replace=False)
    # Set the 'z_spec' values to negative for the selected rows
    train_data.loc[negative_error_indices, 'z_spec'] = -np.abs(train_data.loc[negative_error_indices, 'z_spec'])

In [13]:
# Corruption 5: Introduce random large magnitude values for 0.01% of each of the photometric bands (u, g, r, i, z)
if corruption_bools[4]:
    num_photometric_outliers = int(0.0001 * num_rows)
    for band in ['u', 'g', 'r', 'i', 'z']:
        photometric_outlier_indices = np.random.choice(train_data.index[train_data["is_valid"] == False], size=num_photometric_outliers, replace=False)
        # Add random large values (e.g., between 100 and 200) to the selected photometric band for the selected rows
        train_data.loc[photometric_outlier_indices, f"modelMag_{band}"] += np.random.uniform(50, 100, size=num_photometric_outliers)

In [14]:
# Finally apply exctinction correction to the photometric bands
bands = ["u", "g", "r", "i", "z"]

for band in bands:
    for data in [train_data, test_data]:
        data[band+"_mag"] = (
            data[f"modelMag_{band}"]
            - data[f"extinction_{band}"]
        )

train_data["u-g"] = train_data["u_mag"] - train_data["g_mag"]
train_data["g-r"] = train_data["g_mag"] - train_data["r_mag"]
train_data["r-i"] = train_data["r_mag"] - train_data["i_mag"]
train_data["i-z"] = train_data["i_mag"] - train_data["z_mag"]

train_data["raw_u-g"] = train_data["modelMag_u"] - train_data["modelMag_g"]
train_data["raw_g-r"] = train_data["modelMag_g"] - train_data["modelMag_r"]
train_data["raw_r-i"] = train_data["modelMag_r"] - train_data["modelMag_i"]
train_data["raw_i-z"] = train_data["modelMag_i"] - train_data["modelMag_z"]

train_data["concentration_r"] = (
    train_data["petroR90_r"] / train_data["petroR50_r"]
)
test_data["u-g"] = test_data["u_mag"] - test_data["g_mag"]
test_data["g-r"] = test_data["g_mag"] - test_data["r_mag"]
test_data["r-i"] = test_data["r_mag"] - test_data["i_mag"]
test_data["i-z"] = test_data["i_mag"] - test_data["z_mag"]

test_data["raw_u-g"] = test_data["modelMag_u"] - test_data["modelMag_g"]
test_data["raw_g-r"] = test_data["modelMag_g"] - test_data["modelMag_r"]
test_data["raw_r-i"] = test_data["modelMag_r"] - test_data["modelMag_i"]
test_data["raw_i-z"] = test_data["modelMag_i"] - test_data["modelMag_z"]

test_data["concentration_r"] = (
    test_data["petroR90_r"] / test_data["petroR50_r"]
)

In [15]:
# Students only need colors and spec info:
feature_columns = [
    "source_id",

    # Raw, uncorrected colours
    "raw_u-g",
    "raw_g-r",
    "raw_r-i",
    "raw_i-z",

    # Extinction-corrected colours
    "u-g",
    "g-r",
    "r-i",
    "i-z",

    # Additional optional information
    "r_mag",
    "concentration_r",
]
train_columns = feature_columns + [
    "z_spec",
    "z_spec_err",
    "is_valid",
]

test_columns = [
    "source_id",
    "z_spec",
]

train_data = train_data[train_columns]
student_test_data = test_data[feature_columns]
organizer_test_data = test_data[test_columns]

In [16]:
# Save the corrupted data to a new CSV file
train_data.to_csv('data/train.csv', index=False)
student_test_data.to_csv('data/test.csv', index=False)

In [17]:
solution = organizer_test_data[
    ["source_id", "z_spec"]
].copy()

rng = np.random.default_rng(42)

is_public = (
    rng.random(len(solution)) < 0.50
)

solution["Usage"] = np.where(
    is_public,
    "Public",
    "Private",
)

solution.to_csv(
    "data/solution.csv",
    index=False,
)

print(solution["Usage"].value_counts())
print(
    solution
    .groupby("Usage")["z_spec"]
    .describe()
)

Usage
Private    10062
Public      9938
Name: count, dtype: int64
           count      mean       std       min       25%      50%       75%  \
Usage                                                                         
Private  10062.0  0.297225  0.211535  0.010384  0.105959  0.22762  0.497991   
Public    9938.0  0.297450  0.212080  0.010398  0.104507  0.23244  0.498362   

              max  
Usage              
Private  0.799404  
Public   0.799737  
